In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# INPUT
# -----------------------------
H = float(input("Enter wall height (m): "))

# -----------------------------
# CONSTANT DESIGN RULES
# -----------------------------
base_width = 0.7 * H

# base slope 1:4 (V:H)
theta_base = math.atan(1/4)

band_spacing = 2.0
band_thickness = 0.5

gamma_stone = 22
gamma_cement = 24

# -----------------------------
# FRONT BATTER CHECK
# -----------------------------

# compute natural front angle from geometry assumption
# assume initial top width = 1/5 of base
top_width = base_width / 5

theta_front = math.atan((base_width - top_width) / H)
theta_front_deg = math.degrees(theta_front)

# enforce minimum 10 degree condition
min_angle = math.radians(10)

if theta_front < min_angle:
    theta_front = min_angle
    theta_front_deg = 10

    # recompute top width based on forced angle
    top_width = base_width - H * math.tan(theta_front)

bottom_width = base_width

# -----------------------------
# WIDTH FUNCTION
# -----------------------------
def width_at(y):
    return top_width + (bottom_width - top_width) * (1 - y / H)

# -----------------------------
# VOLUME OF WALL (TRAPEZOID)
# -----------------------------
total_area = 0.5 * (top_width + bottom_width) * H
total_volume = total_area * 1  # per meter length

# -----------------------------
# HORIZONTAL CEMENT BANDS
# -----------------------------
num_h_bands = int(H / band_spacing) + 1
h_band_volume = num_h_bands * band_thickness * ((top_width + bottom_width)/2)

# -----------------------------
# VERTICAL BANDS (PLAN GRID ASSUMPTION)
# -----------------------------
length = 1.0  # per meter run
num_v_bands = int(length / band_spacing) + 1

v_band_volume = num_v_bands * band_thickness * H

# -----------------------------
# TOTAL BAND VOLUME
# -----------------------------
cement_volume = h_band_volume + v_band_volume
stone_volume = total_volume - cement_volume

# -----------------------------
# WEIGHT
# -----------------------------
stone_weight = stone_volume * gamma_stone
cement_weight = cement_volume * gamma_cement
total_weight = stone_weight + cement_weight

# -----------------------------
# OUTPUT
# -----------------------------
print("\n----- DESIGN SUMMARY -----")
print(f"Height: {H} m")
print(f"Base width (0.7H): {base_width:.2f} m")
print(f"Top width: {top_width:.2f} m")

print(f"Front batter angle: {theta_front_deg:.2f}°")
print(f"Base slope: 1:4 (fixed)")

print("\n----- VOLUMES -----")
print(f"Total volume: {total_volume:.3f} m3")
print(f"Cement volume: {cement_volume:.3f} m3")
print(f"Stone volume: {stone_volume:.3f} m3")

print("\n----- WEIGHT -----")
print(f"Cement weight: {cement_weight:.2f} kN")
print(f"Stone weight: {stone_weight:.2f} kN")
print(f"Total weight: {total_weight:.2f} kN")

# -----------------------------
# PLOT
# -----------------------------
plt.figure(figsize=(6, 8))

y_vals = np.linspace(0, H, 200)
x_left = [0]*len(y_vals)
x_right = [width_at(y) for y in y_vals]

plt.plot(x_left, y_vals, 'k')
plt.plot(x_right, y_vals, 'k')

# fill layers
step = 0.1
y = 0

while y < H:
    y2 = min(y + step, H)
    y_mid = (y + y2) / 2

    if int(y_mid // band_spacing) * band_spacing <= y_mid < int(y_mid // band_spacing) * band_spacing + band_thickness:
        color = "#8b5a2b"
    else:
        color = "#d2b48c"

    plt.fill(
        [0, width_at(y), width_at(y2), 0],
        [y, y, y2, y2],
        color=color,
        edgecolor="black",
        linewidth=0.2
    )

    y = y2

plt.title("Banded Retaining Wall (Constrained Design)")
plt.xlabel("Width (m)")
plt.ylabel("Height (m)")
plt.axis("equal")
plt.grid(True)

plt.show()